<a href="https://colab.research.google.com/github/KevinJoseph-DataPortfolio/KevinJoseph-DataAnalyticsPortfolio/blob/main/03_AI_DataCleaning_Agent/cleaning_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# ── 1. CHARGEMENT ──────────────────────────────────────────────────────────────
url = "https://raw.githubusercontent.com/KevinJoseph-DataPortfolio/KevinJoseph-DataAnalyticsPortfolio/main/03_AI_DataCleaning_Agent/lab_quality_data_RAW_EN.csv"
df_raw = pd.read_csv(url)
df = df_raw.copy()

print(f"✅ Fichier chargé : {len(df)} lignes, {len(df.columns)} colonnes")

# ── 2. RAPPORT AVANT NETTOYAGE ─────────────────────────────────────────────────
report = {}
report["rows_before"] = len(df)
report["duplicates"] = df.duplicated().sum()
report["missing_before"] = df.isnull().sum().sum()
report["quality_score_before"] = round((1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100, 1)

# ── 3. NETTOYAGE ───────────────────────────────────────────────────────────────

# Doublons
df = df.drop_duplicates(subset=["Sample_ID"], keep="first")

# Casse : uniformiser en Title Case les colonnes texte
for col in ["Product", "Category", "Production_Site", "Compliance_Status", "Operator"]:
    df[col] = df[col].str.strip().str.title()

# Unités : normaliser
unit_map = {
    "mg/l": "mg/L", "mg/l": "mg/L", "MG/L": "mg/L",
    "g/l": "g/L", "G/L": "g/L",
    "ug/L": "µg/L", "ug/l": "µg/L", "µg/l": "µg/L", "UG/L": "µg/L",
    "cfu/ml": "CFU/mL", "CFU/ml": "CFU/mL", "cfu/mL": "CFU/mL",
    "PPM": "ppm", "Ppm": "ppm"
}
df["Unit_of_Measure"] = df["Unit_of_Measure"].replace(unit_map)

# Dates : standardiser en YYYY-MM-DD
def parse_date(d):
    if pd.isnull(d):
        return np.nan
    for fmt in ["%d/%m/%Y", "%Y-%m-%d", "%m-%d-%Y", "%d.%m.%Y", "%d-%m-%Y"]:
        try:
            return datetime.strptime(str(d).strip(), fmt).strftime("%Y-%m-%d")
        except:
            continue
    return np.nan

df["Collection_Date"] = df["Collection_Date"].apply(parse_date)

# Températures : extraire le chiffre uniquement
def clean_temp(t):
    if pd.isnull(t):
        return np.nan
    match = re.search(r"\d+", str(t))
    return int(match.group()) if match else np.nan

df["Storage_Temperature"] = df["Storage_Temperature"].apply(clean_temp)

# Valeurs aberrantes numériques
df["Bacteriological_Result"] = pd.to_numeric(df["Bacteriological_Result"], errors="coerce")
df.loc[df["Bacteriological_Result"] < 0, "Bacteriological_Result"] = np.nan
df.loc[df["Bacteriological_Result"] > 2000, "Bacteriological_Result"] = np.nan

df["pH_Level"] = pd.to_numeric(df["pH_Level"], errors="coerce")
df.loc[df["pH_Level"] < 0, "pH_Level"] = np.nan
df.loc[df["pH_Level"] > 14, "pH_Level"] = np.nan

df["Chemical_Result"] = pd.to_numeric(df["Chemical_Result"], errors="coerce")

# Valeurs manquantes : médiane pour numérique, "Unknown" pour texte
for col in ["Bacteriological_Result", "Chemical_Result", "pH_Level", "Storage_Temperature"]:
    df[col] = df[col].fillna(df[col].median())

for col in ["Product", "Category", "Production_Site", "Compliance_Status", "Operator", "Unit_of_Measure", "Batch_Number"]:
    df[col] = df[col].fillna("Unknown")

# ── 4. RAPPORT APRÈS NETTOYAGE ─────────────────────────────────────────────────
report["rows_after"] = len(df)
report["rows_removed"] = report["rows_before"] - report["rows_after"]
report["missing_after"] = df.isnull().sum().sum()
report["quality_score_after"] = round((1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100, 1)

print("\n" + "="*55)
print("       LAB DATA CLEANING REPORT — KÉVIN JOSEPH")
print("="*55)
print(f"  Rows before cleaning      : {report['rows_before']:,}")
print(f"  Rows after cleaning       : {report['rows_after']:,}")
print(f"  Duplicates removed        : {report['rows_removed']:,}")
print(f"  Missing values before     : {report['missing_before']:,}")
print(f"  Missing values after      : {report['missing_after']:,}")
print(f"  Data quality score BEFORE : {report['quality_score_before']}%")
print(f"  Data quality score AFTER  : {report['quality_score_after']}%")
print("="*55)

# ── 5. EXPORT ──────────────────────────────────────────────────────────────────
df.to_csv("lab_quality_data_CLEAN.csv", index=False)
print("\n✅ Fichier propre exporté : lab_quality_data_CLEAN.csv")

✅ Fichier chargé : 10000 lignes, 13 colonnes

       LAB DATA CLEANING REPORT — KÉVIN JOSEPH
  Rows before cleaning      : 10,000
  Rows after cleaning       : 9,506
  Duplicates removed        : 494
  Missing values before     : 11,349
  Missing values after      : 0
  Data quality score BEFORE : 91.3%
  Data quality score AFTER  : 100.0%

✅ Fichier propre exporté : lab_quality_data_CLEAN.csv
